<a href="https://colab.research.google.com/github/JuliaLoth/slimmermetai/blob/main/Substack_note_creator_telegram_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install anthropic
!pip install feedparser
!pip install python-telegram-bot
!pip install nest_asyncio python-telegram-bot --upgrade

In [9]:
!pip install nest_asyncio python-telegram-bot --upgrade

import nest_asyncio
nest_asyncio.apply()  # Past de bestaande event loop in Colab aan

import io
import sys
import logging
import asyncio
import datetime
from dateutil import parser
import feedparser
import anthropic
from google.colab import userdata  # Zorg dat dit werkt in Colab

# Telegram imports (python-telegram-bot v20+)
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, ContextTypes

# Haal de Telegram API key op uit Colab secrets (gebruik de naam TELEGRAM_API_KEY)
try:
    TELEGRAM_API_KEY = userdata.get('TELEGRAM_API_KEY')
except Exception as e:
    print("Kan TELEGRAM_API_KEY niet ophalen via Colab secrets, gebruik een handmatig ingestelde token voor testen.")
    TELEGRAM_API_KEY = "JOUW_TELEGRAM_TOKEN_HIER"

if not TELEGRAM_API_KEY:
    print("❌ Telegram API key niet gevonden. Voeg deze toe als een Colab secret met de naam 'TELEGRAM_API_KEY'.")

# Stel logging in (handig voor debuggen)
logging.basicConfig(
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

# ----------------------- NIEUWS-SCRIPT FUNCTIES -----------------------

def fetch_news_from_rss(rss_url):
    """Haalt nieuwsitems op uit een RSS-feed."""
    feed = feedparser.parse(rss_url)
    return feed.get('entries', [])

def is_relevant_for_target_audience(entry, anthropic_client):
    """Bepaalt of een nieuwsitem relevant is voor de doelgroep met behulp van Claude."""
    titel = entry.title
    summary = entry.summary if hasattr(entry, 'summary') else "Geen samenvatting beschikbaar."
    filter_prompt = f"""Beoordeel de relevantie van het volgende nieuwsitem voor de 'Slimmer met AI' nieuwsbrief,
die gericht is op Nederlandse professionals (MBO4-WO, digitaal vaardig, geen tech-experts)
die praktische AI-toepassingen zoeken om slimmer te werken. Het maakt niet uit of het artikel in het engels is.

'Slimmer met AI' richt zich op:
- Praktische AI tools en toepassingen voor dagelijks werk
- Tijdsbesparing en efficiëntie met AI
- AI voor niet-techneuten, helder en toegankelijk uitgelegd
- Nederlandse context en voorbeelden

Nieuwsitem:
Titel: {titel}
Samenvatting: {summary}

Vraag: Is dit nieuwsitem **RELEVANT** voor de 'Slimmer met AI' doelgroep?
Antwoord met alleen 'JA' of 'NEE'."""
    try:
        response = anthropic_client.messages.create(
            model="claude-3-opus-20240229",
            max_tokens=10,
            messages=[{"role": "user", "content": filter_prompt}]
        )
        relevancy_answer = response.content[0].text.strip().upper()
        return "JA" in relevancy_answer
    except Exception as e:
        logger.error(f"Fout bij relevantie check: {e}")
        return False

def create_substack_note(entry, anthropic_client):
    """Genereert een Substack note in 'Slimmer met AI' stijl met de Claude API voor een nieuwsitem."""
    titel = entry.title
    link = entry.link
    summary = entry.summary if hasattr(entry, 'summary') else "Geen samenvatting beschikbaar."

    prompt_claude = f"""Je bent een AI-assistent genaamd Julia, de 'Tech-Expert die andere professionals graag helpt door slimmer te werken en vooral niet harder. Jij vindt shortcuts in het werk doormiddel van AI, geeft graag tips en adviezen en verrassende en originele inzichten' voor 'Slimmer met AI', een Substack nieuwsbrief voor Nederlandse professionals over AI.

Genereer een korte, pakkende Substack note over het volgende nieuwsitem, gericht op Nederlandse professionals (MBO4-WO, digitaal vaardig, geen tech-experts).

Stijlkenmerken 'Slimmer met AI':
- Geestige en droge Tech-Expert: Nuchtere Expert (45%), Betrokken Collega (30%), Zelfspot & Relativering (25%)
- Persoonlijke Authenticity: 'Van moeilijk en hard werken naar simpel en controle dankzij tech', 'Automatiseer uit luiheid (en dat is oké!)', 'Eigen fails als leermomenten', 'Tech als redding voor de drukke en soms chaotische professional'
- Signature Zelfspot & Schrijfstijl: Kort en krachtig, direct en persoonlijk, informeel maar professioneel, concreet en praktisch, met een kwinkslag.
- Toon: "Hé, ik snap dat AI best ingewikkeld kan zijn. Maar geen zorgen - we hakken dit in behapbare stukken! 💪"
- Advies: "Uit ervaring weet ik dat dit vaak werkt: [praktische tip]. Maar test vooral wat voor jou het beste werkt!"
- Humor: "Herkenbaar? Je zit weer eens een rapport te typen dat ChatGPT in 5 minuten had kunnen doen... 😅"
- Doelgroep: Nederlandse early majority professionals (25-54, MBO4-WO, digitaal vaardig, praktische AI toepassingen)

Nieuwsitem:
Titel: {titel}
Link: {link}
Samenvatting: {summary}

Genereer NU de Substack note in MARKDOWN formaat. Zorg ervoor dat de originele link duidelijk wordt weergegeven zodat ik deze kan linken in mijn post."""

    try:
        response = anthropic_client.messages.create(
            model="claude-3-opus-20240229",
            max_tokens=500,
            messages=[{"role": "user", "content": prompt_claude}]
        )
        return response.content[0].text
    except Exception as e:
        return f"**Fout bij genereren van note met Claude:** {e}\nTitel: {titel}\nLink: {link}\nSamenvatting: {summary}"

def main():
    """
    Haalt nieuws op en genereert Substack notes met de Claude API.
    Retourneert een lijst met de uiteindelijke, gegenereerde notes.
    """
    anthropic_api_key = userdata.get('Anthropic_API_KEY')
    if not anthropic_api_key:
        print("❌ Anthropic API key niet gevonden. Voeg deze toe als een Colab secret met de naam 'Anthropic_API_KEY'.")
        return []
    anthropic_client = anthropic.Anthropic(api_key=anthropic_api_key)
    rss_feeds = [
        "https://www.emerce.nl/rss",
        "https://www.frankwatching.com/feed/",
        "https://www.theverge.com/tech/rss/index.xml",
        "https://medium.com/feed/topic/artificial-intelligence",
        "https://www.marktechpost.com/feed",
        "https://dutch-tech.nl/feed",
        "https://dtx.nl/feed",
        "https://siliconcanals.com/feed",
        "https://netherlandsinnovation.nl/feed",
        "https://bronict.nl/rss-feed/"
    ]
    final_notes = []
    today = datetime.datetime.now(datetime.timezone.utc)
    one_week_ago = today - datetime.timedelta(weeks=1)
    for rss_url in rss_feeds:
        entries = fetch_news_from_rss(rss_url)
        if not entries:
            continue
        for entry in entries:
            published_date = None
            if hasattr(entry, 'published'):
                try:
                    published_date = parser.parse(entry.published)
                except:
                    published_date = today
            elif hasattr(entry, 'updated'):
                try:
                    published_date = parser.parse(entry.updated)
                except:
                    published_date = today
            if published_date and published_date < one_week_ago.replace(tzinfo=datetime.timezone.utc):
                continue
            if is_relevant_for_target_audience(entry, anthropic_client):
                note = create_substack_note(entry, anthropic_client)
                final_notes.append(note)
    return final_notes

def run_news_script():
    """Voert de main() functie uit en retourneert de lijst met gegenereerde notes."""
    return main()

# ----------------------- TELEGRAM BOT INTEGRATIE -----------------------

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("Hallo! Ik ben de 'Slimmer met AI' bot. Stuur /genereer om nieuws op te halen en Substack notes te genereren.")

async def genereer(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("Even geduld aub, ik haal het nieuws op en genereer de Substack notes...")
    notes = run_news_script()
    if notes:
        for note in notes:
            await update.message.reply_text(note, parse_mode="Markdown")
    else:
        await update.message.reply_text("Geen relevante Substack notes gegenereerd.")

async def main_telegram():
    app = ApplicationBuilder().token(TELEGRAM_API_KEY).build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("genereer", genereer))
    logger.info("Bot draait... (Let op: Colab notebooks zijn niet ideaal voor 24/7 draaien)")
    await app.initialize()
    await app.run_polling()

# Start de bot op de achtergrond zodat de cell niet blokkeert
asyncio.create_task(main_telegram())
print("Telegram bot draait op de achtergrond!")
